# Practical session: Introduction to biopython and applications in structural bioinformatics

**What is biopython?**

Biopython is a set of freely available tools for biological computation written in Python by an international team of developers. It is a distributed collaborative effort to develop Python libraries and applications which address the needs of current and future work in bioinformatics. 

**Why should we use it?**

An immense amount of biological data is being generated and published constantly, and being able to interact with this data programatically will enable us to perform analysis on a larger scale. In particular, with the advent of Machine Learning and Big Data, having the ability to access and process biological datasets efficiently is fundamental to integrate these tools into our workflows.

**What are we going to do with biopython?**

Here, we will become familiar with some of the main modules of biopython and their applications. In particular, we will explore the Bio.PDB module and solve some exercises, to get hands-on experience with the tools that biopython offers to parse and analyze protein structures. 

In [ ]:
# Uncomment the following line if you need to install dependencies
#!pip install biopython matplotlib
# You should have biopython >= 1.83

In [ ]:
# Load modules
import Bio
import numpy as np
import matplotlib.pyplot as plt

## Sequence modules overview

### Bio.Seq

This module provides objects to represent biological sequences with alphabets.

In [ ]:
# Load modules
from Bio.Seq import Seq

In [ ]:
# Let us define a dummy DNA sequence
dna = Seq("ATGCGTACG")
print(f'Original sequence:        {dna}')
# We can obtain its reverse complement (DNA and RNA)
print(f'Reverse complement:       {dna.reverse_complement()}')
print(f'Reverse complement (RNA): {dna.reverse_complement_rna()}')
# As well as its translation
print(f'\nTranslated sequence: {dna.translate()}')

In [ ]:
# Let us define a dummy protein sequence
prot = Seq("HHARARMLLIVAAR")

# We can easily count a given amino acid
print('Number of A:', prot.count('A'))

# And a given pattern
print('Number of AR:', prot.count('AR'))

In general, many methods from the Python string class are also implemented for Seq.

### Bio.SeqIO

In [ ]:
# Load modules
from Bio import SeqIO

In [ ]:
# Read information from a FASTA file
path = './Data/ACE2_YEAST.fasta'
for record in SeqIO.parse(path, "fasta"):
    print(record.id)
    print(record.seq)

In this way, we can stream large datasets without loading them fully into memory.

In [ ]:
# Read multiple records in plain text Swiss-Prot aka UniProt format.
path = './Data/uniprotkb_yeast_transcription_factor_AN_2026_03_10.txt'
records = SeqIO.to_dict(SeqIO.parse(path, 'swiss'))
print(f'Identified {len(records)} entries\n')

# We can now access the data by uniprot id
unip_id = 'P11747'
print('Example entry:')
print('UniProt ID:', unip_id)
print('Name:', records[unip_id].name)
print('Sequence:', records[unip_id].seq)

## The Bio.PDB module

### Fetching data from PDB

In [ ]:
# Load modules
from Bio.PDB import PDBList

In [ ]:
# Let us fetch some structures we will use later
pdbl = PDBList(server="https://files.rcsb.org")
pdbl.retrieve_pdb_file('1rkl', pdir='./Data/', file_format='mmCif')
pdbl.retrieve_pdb_file('3goe', pdir='./Data/', file_format='mmCif')
pdbl.retrieve_pdb_file('1joy', pdir='./Data/', file_format='mmCif')
pdbl.retrieve_pdb_file('1fk9', pdir='./Data/', file_format='mmCif')

### Reading structures from PDB and mmCIF

In [ ]:
# Load modules
from Bio.PDB import PDBParser, MMCIFParser

In [ ]:
# Initialize parser
parser = PDBParser()
# Read our PDB structure
structure = parser.get_structure("ACE2_yeast", "./Data/AF-P21192-F1-model_v6.pdb")

# Let's investigate the structure of the object
print('Structure (ACE2_YEAST):')
print('-> Models:', list(structure))
print('   -> Chains:', list(structure[0]))
print('      -> Residues:', len(structure[0]['A']))
print('         -> Atoms:', len(list(structure[0]['A'].get_atoms())))

In [ ]:
# Initialize parser
parser = MMCIFParser()
# Read our mmCIF structure
structure = parser.get_structure("1rkl", "./Data/1rkl.cif")

# Let's investigate the structure of the object
print('Structure (NMR structure of yeast oligosaccharyltransferase subunit Ost4p):')
print('-> Models:', len(list(structure)))
print('   -> Chains:', len(list(structure[0])))
print('      -> Residues:', len(structure[0]['A']))
print('         -> Atoms:', len(list(structure[0]['A'].get_atoms())))

### Chain discontinuity warning

In [ ]:
# Initialize parser
parser = MMCIFParser()
# Read our mmCIF structure with missing residues
structure = parser.get_structure("1fk9", "./Data/1fk9.cif")

# If we try to iterate over the residues we will get a warning
residues = list(structure[0].get_residues())

### Separating proteins from ions / ligands / water

In [ ]:
# How can we understand if a residue is representing a heteroatom?
all_ids = []
for i, r in enumerate(residues):
    all_ids.append(r.get_id()[0])

print('Unique residue IDs in 1fk9:')
print(np.unique(all_ids))

We have H_CSD for a PTM residue, H_EFZ for the EFAVIRENZ ligand and W for water.

In [ ]:
# Now we can extract protein residues
prot_residues = []
for r in residues:
    # Skip H_EFZ and W residues
    if r.get_id()[0] in ['H_EFZ', 'W']:
        continue

    prot_residues.append(r)

### Adding custom properties to a model

In [ ]:
from Bio.PDB import MMCIFParser, MMCIFIO

In [ ]:
# Initialize parser
parser = MMCIFParser()
# Read our mmCIF structure
structure = parser.get_structure("1rkl", "./Data/1rkl.cif")

# Show the current B-factor values for the first atoms in
# the first model
print('Original B-factor values:')
for i, a in enumerate(structure[0].get_atoms()):
    print(i, a.get_bfactor())
    if i >= 4:
        break

# Assume we have some computed values we want to assign to 
# our atoms in the B-factor attribute
custom_bfactor = np.random.uniform(size=len(list(structure[0].get_atoms())))

# Assign the new values with set_bfactor(), making sure we 
# iterate over all models for consistency
for mdl in structure:
    for i, a in enumerate(structure[0].get_atoms()):
        a.set_bfactor(custom_bfactor[i])

# Print new values for the same atoms as before
print('\nCustom B-factor values:')
for i, a in enumerate(structure[0].get_atoms()):
    print(i, a.get_bfactor())
    if i >= 4:
        break

# We can write the structure with our custom B-factor to
# a new file
io = MMCIFIO()
io.set_structure(structure)
io.save('./Data/1rkl_custom.cif')

In [ ]:
# If we now read the saved file, we should find our custom values
parser = MMCIFParser()
structure = parser.get_structure("1rkl_custom", "./Data/1rkl_custom.cif")

print('Custom B-factor values:')
for i, a in enumerate(structure[0].get_atoms()):
    print(i, a.get_bfactor())
    if i >= 4:
        break

### Extracting the residue sequence from a structure

In [ ]:
# Load modules
from Bio.PDB.Polypeptide import PPBuilder

# Dictionary to map between the 3-letter and 1-letter codes for amino acids
d3to1 = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
 'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N', 
 'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W', 
 'ALA': 'A', 'VAL':'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

In [ ]:
# Let us load the 3GOE PDB entry
parser = MMCIFParser()
structure = parser.get_structure("3goe", "./Data/3goe.cif")

# What residues do we have in the structure?
all_residues = [r.get_resname() for r in structure[0]['A'].get_residues()]
print(np.unique(all_residues))

Notice that we have 'CA' (ion) and 'HOH' (water) residues.

In [ ]:
# Build sequence naively
seq = ''.join([d3to1[r] if r in d3to1 else 'X' for r in all_residues])
print('Sequence (including ions and water):\n', seq)

We could build a simple logic to remove the ions and water from the sequence, but let us leverage the functions already implemented in biopython...

In [ ]:
# Build sequence with biopython
ppb = PPBuilder()
for pp in ppb.build_peptides(structure):
    print(pp.get_sequence())

### Analyzing the sequence

In [ ]:
# Load modules
from Bio.SeqUtils.ProtParam import ProteinAnalysis

In [ ]:
# Get our sequence
ppb = PPBuilder()
seq = ppb.build_peptides(structure)[0].get_sequence()
seq_analysis = ProteinAnalysis(seq)

# Let us print the amino acid percentages
print('Amino acid percentages for entry 3GOE:')
print(seq_analysis.get_amino_acids_percent())

# We could also, for instance, calculate the aromaticity
# of the sequence, according to Lobry, 1994: P(F + W + Y)
print(f'\nAromaticity for entry 3GOE: {seq_analysis.aromaticity():.4f}')

# Additionally, we can get the grand average of hydropathy, 
# as defined by Kyte and Doolittle (1982)
print(f'\nGRAVY for entry 3GOE: {seq_analysis.gravy():.2f} (strongly hydrophilic)')

In [ ]:
# We must be careful and understand the analysis we are running
# For instance, let us compute the secondary structure fraction
# for ACE2_yeast:
parser = PDBParser()
structure = parser.get_structure("ACE2_yeast", "./Data/AF-P21192-F1-model_v6.pdb")

ppb = PPBuilder()
seq = ppb.build_peptides(structure)[0].get_sequence()
seq_analysis = ProteinAnalysis(seq)

print('Secondary structure fraction of ACE2_YEAST:')
sec_struct_fraction = seq_analysis.secondary_structure_fraction()
for i, s in enumerate(['Helix', 'Turn', 'Sheet']):
    print(f'{s}: {sec_struct_fraction[i]:.2f} %')

By reading at the documentation, it can be noted that the secondary structure fraction is estimated here by simply grouping the amino acids into categories. However, if we look at the structure (https://alphafold.ebi.ac.uk/entry/P21192) we can see that it is predicted to be mostly disordered...

The reader is highly encouraged to check all the other utilities that are included in ```Bio.SeqUtils```.

### Calculating distances

In [ ]:
# Let us load the 3GOE PDB entry
parser = MMCIFParser()
structure = parser.get_structure("3goe", "./Data/3goe.cif")

In [ ]:
# We can access the carbon alpha for each residue
ca1 = structure[0]["A"][1]["CA"]
ca2 = structure[0]["A"][80]["CA"]

# Biopython implements distance calculation internally
dist = ca1 - ca2
print('Distance (biopython):', dist)

# We can check that it works properly using numpy
dist_np = np.linalg.norm(ca1.get_coord() - ca2.get_coord())
print('Distance (numpy):    ', dist_np)

**Exercise:** Extract the carbon alpha coordinates of the protein without the N-terminal histidine tag (first 6 residues) nor the water and ion atoms (hetero atoms).

In [ ]:
# Place the results into the coords variable as 
# an array of shape (N_residues, 3)
coords = None

# Write your code here:



In [ ]:
# Run this cell to check if your output is correct
if np.array(coords).shape != (74, 3):
    print('The result is incorrect: array has the wrong shape')
else:
    ex_check = np.prod(np.isclose(
        np.array(coords), 
        np.loadtxt('./Data/3GOE_CA_coords_without_H_and_hetero.txt')))
    if ex_check:
        print('The result is correct')
    else:
        print('The result is incorrect: the values do not match the expected outcome')

In [ ]:
# Uncomment the following line to see the expected outcome
#np.loadtxt('./Data/3GOE_CA_coords_without_H_and_hetero.txt')

**Exercise:** Create a all-to-all contact map for the protein residues using the coordinates array we generated previously. Use 6 A and 12 A as the inner and outer cutoffs, respectively.

In [ ]:
# Place the contact map into the M variable as an array of
# shape (N_residues, N_residues)
M = None

# Write your code here:


In [ ]:
# Run this cell to check if your output is correct
if np.array(M).shape != (74, 74):
    print('The result is incorrect: array has the wrong shape')
else:
    ex_check = np.prod(np.isclose(
        np.array(M), 
        np.loadtxt('./Data/3GOE_contact_map_without_H_and_hetero.txt')))
    if ex_check:
        print('The result is correct')
    else:
        print('The result is incorrect: the values do not match the expected outcome')

In [ ]:
# Uncomment the following line to see the expected outcome
#np.loadtxt('./Data/3GOE_contact_map_without_H_and_hetero.txt')

In [ ]:
# We can plot the contact matrix, just use the following lines
#plt.imshow(M)
#plt.show();

### Superimposing multiple models

In [ ]:
# Load modules
from Bio.PDB import MMCIFParser, Superimposer

In [ ]:
# Let us load the 1JOY PDB entry
parser = MMCIFParser()
structure = parser.get_structure("1joy", "./Data/1joy.cif")

# How many models do we have?
print(f'Structure has {len(list(structure))} models')

In [ ]:
# Let us align the last 20 models to the first one:
# Initialize superimposer
superimposer = Superimposer()

# Set reference model
ref_model = structure[0]
print(f'RMS values using model {ref_model.id} as reference:')
for model in structure:
    # Skip model if it is the same as the ref.
    if model.id == ref_model.id:
        continue

    # Get CA atoms
    ref_atoms = [res['CA'] for res in ref_model.get_residues()]
    alt_atoms = [res['CA'] for res in model.get_residues()]

    # Align the alternative atoms
    superimposer.set_atoms(ref_atoms, alt_atoms)
    superimposer.apply(model.get_atoms())

    print(f'Model {model.id}: {superimposer.rms:.2f}')

In [ ]:
# Now, we can estimate the RMSD for each residue
res_rmsd_A = dict()
res_rmsd_B = dict()
for chain in ref_model:
    for res in chain.get_residues():
        # Initialize deviation values for each residue
        res_deviations = []
        
        for model in structure:
            # Skip reference model
            if model.id == ref_model.id:
                continue
    
            # Get alt. chain and residue
            alt_chain = [ch for ch in model if ch.id == chain.id][0]
            alt_res = [r for r in alt_chain.get_residues() if r.id == res.id][0]
    
            # Get deviation (here we only consider CA positions)
            res_deviations.append(alt_res['CA'] - res['CA'])
    
        # Calculate residue RMSD and save for each chain
        if chain.id == 'A':
            res_rmsd_A[res.id] = np.sqrt(np.mean(np.square(res_deviations)))
        else:
            res_rmsd_B[res.id] = np.sqrt(np.mean(np.square(res_deviations)))

In [ ]:
# Finally, we can plot the RMSD per residue
FONTSIZE = 13
fig, ax = plt.subplots(figsize=(5,3), nrows=1, ncols=1, dpi=200)

# Plots
plt.plot(res_rmsd_A.values(), label='Chain A')
plt.plot(res_rmsd_B.values(), label='Chain B')

# Labels and legend
plt.ylabel(r'RMSD', fontsize=FONTSIZE)
plt.xlabel(r'Position in sequence', fontsize=FONTSIZE)
plt.tick_params(labelsize=FONTSIZE-2)
plt.legend(fontsize=FONTSIZE-2)

# Limits
plt.xlim(left=0, right=max([len(res_rmsd_A)-1, 
                            len(res_rmsd_B)-1]))

plt.show();

### Calculating geometric information

In [ ]:
# Let us read two different structures to compare their geometry:
# First, let us parse the structure files
parser = MMCIFParser()
struct_1 = parser.get_structure("3goe", "./Data/3goe.cif")
struct_2 = parser.get_structure("1joy", "./Data/1joy.cif")

# We will only use the first model for each
model_1 = struct_1[0]
model_2 = struct_2[0]

# Now, we need to define a function to estimate the radius
# of gyration for a given model
def get_radius_of_gyration(mdl):
    # Get residues, excluding water, ligands and ions
    res = [r for r in mdl.get_residues() if r.get_id()[0] == ' ']

    # Extract coordinates from the CA of each residue
    X = np.array([r['CA'].get_coord() for r in res])

    # Get the geometric center of mass
    r_cm = np.mean(X, axis=0)

    # Calculate the radius of gyration and return
    R2_G = np.mean(np.square(np.linalg.norm(X - r_cm, axis=0)))
    return R2_G

# Finally, we get our results
R2_G_1 = get_radius_of_gyration(model_1)
R2_G_2 = get_radius_of_gyration(model_2)


print('Radius of gyration:')
print(f'Entry 3GOE: {str(int(R2_G_1)).rjust(6)} A^2')
print(f'Entry 1JOY: {str(int(R2_G_2)).rjust(6)} A^2')

This agrees with what we have observed qualitatively, that 3GOE is globular while 1JOY has a more extended configuration.

## Hands-on exercise: human hemoglobin

1. Download the mmCIF file from PDB (6FQF) and parse it.
2. Extract the sequence of the protein subunits and obtain the overall amino acid composition
3. Identify for each chain the residue that is closest to its corresponding iron atom
    - Remember to account for all the atoms in each residue, not only the C$\alpha$
4. Calculate the radius of gyration of the individual chains, as well as for the complete structure
    - You can use the function we defined previously
6. _(Optional)_ Compare the structure with that of hemoglobin when bound to oxygens (PDB: 1GZX)